# SXM Batch Processing to GWY Format

This notebook demonstrates how to use the `sxm_batch_processor` module to batch process .sxm files and convert them to .gwy format.

## Features
- Find all .sxm files in a directory tree
- Apply processing operations:
  - Level data by mean plane subtraction
  - Align rows (mean, median, or match height methods)
  - Parabolic background subtraction
  - Remove scars
- Save to .gwy format preserving metadata
- Files are saved in the same location as the original .sxm files

## Import the module

In [1]:
import sxm_batch_processor as sbp
from pathlib import Path

## Example 1: Find all .sxm files in a directory

First, let's see what files we have:

In [2]:
# Specify your data directory
data_directory = "D:/PORT/2dataProcessing/test_data"

# Find all .sxm files
sxm_files = sbp.find_sxm_files(data_directory)

print(f"Found {len(sxm_files)} .sxm files:")
for f in sxm_files[:10]:  # Show first 10
    print(f"  {f}")

Found 3 .sxm files:
  D:\PORT\2dataProcessing\test_data\scan_003.sxm
  D:\PORT\2dataProcessing\test_data\subfolder\scan_001.sxm
  D:\PORT\2dataProcessing\test_data\subfolder\scan_002.sxm


## Example 2: Process a single file

Process one file with default settings:

# Process a single file
if len(sxm_files) > 0:
    input_file = sxm_files[0]
    output_file = sbp.sxm_to_gwy(
        input_file,
        level_plane=True,
        align_rows='mean',
        parabolic_sub=True,
        remove_scars_flag=True,
        scar_threshold=3.0
    )
    print(f"Processed: {input_file.name}")
    print(f"Saved to: {output_file}")

## Example 3: Batch process entire directory

Process all .sxm files in a directory and its subdirectories:

In [3]:
# Batch process with default settings
created_files = sbp.batch_process_directory(
    data_directory,
    level_plane=True,
    align_rows='mean',
    parabolic_sub=True,
    remove_scars_flag=True,
    scar_threshold=3.0,
    verbose=True
)

print(f"\nSuccessfully created {len(created_files)} .gwy files")

Found 3 .sxm files
Processing 1/3: scan_003.sxm
  -> Saved to scan_003.gwy
Processing 2/3: scan_001.sxm
  -> Saved to scan_001.gwy
Processing 3/3: scan_002.sxm


D:\MOQUZ\Documents\APROJECTS\sxm-batch-process-save-to-gwy\sxm_batch_processor.py:90: RuntimeWarning: Mean of empty slice
  row_mean = np.nanmean(data[i, :])
D:\MOQUZ\Documents\APROJECTS\sxm-batch-process-save-to-gwy\sxm_batch_processor.py:210: RuntimeWarning: Mean of empty slice
  row_diff_means = np.nanmean(row_diffs, axis=1)


  -> Saved to scan_002.gwy

Completed: 3/3 files processed successfully

Successfully created 3 .gwy files


## Example 4: Batch process with custom settings

You can customize the processing pipeline:

# Process with median row alignment instead of mean
created_files = sbp.batch_process_directory(
    data_directory,
    level_plane=True,
    align_rows='median',  # Use median instead of mean
    parabolic_sub=True,
    remove_scars_flag=True,
    scar_threshold=2.5,  # More sensitive scar detection
    verbose=True
)

## Example 5: Disable certain processing steps

# Only level and align, skip parabolic subtraction and scar removal
created_files = sbp.batch_process_directory(
    data_directory,
    level_plane=True,
    align_rows='mean',
    parabolic_sub=False,  # Skip parabolic subtraction
    remove_scars_flag=False,  # Skip scar removal
    verbose=True
)

## Example 6: Verify output files can be read by gwyfile

Verify that the created .gwy files can be read and processed:

In [4]:
import gwyfile

if len(created_files) > 0:
    # Read the first created .gwy file
    gwy_file = created_files[0]
    gwy_obj = gwyfile.load(str(gwy_file))

    print(f"Successfully loaded: {gwy_file.name}")
    print(f"\nChannels in file:")

    # List all channels
    for key in gwy_obj.keys():
        if key.endswith('/data'):
            datafield = gwy_obj[key]
            title_key = key.replace('/data', '/data/title')
            title = gwy_obj.get(title_key, 'Unknown')
            print(f"  {title}: {datafield.data.shape}")

    # Show metadata
    print(f"\nMetadata keys:")
    for key in gwy_obj.keys():
        if key.endswith('/meta'):
            meta = gwy_obj[key]
            for meta_key in list(meta.keys())[:5]:  # Show first 5
                print(f"  {meta_key}: {meta[meta_key]}")

Successfully loaded: scan_003.gwy

Channels in file:
  Z(forward): (512, 512)
  Z(forward): (512, 512)

Metadata keys:
  nanonis_version: 50
  scanit_type: FLOAT            MSBFIRST
  rec_date: 30.12.2022
  rec_time: 19:34:43
  rec_temp: 290.0000000000
  nanonis_version: 50
  scanit_type: FLOAT            MSBFIRST
  rec_date: 30.12.2022
  rec_time: 19:34:43
  rec_temp: 290.0000000000


## Available Row Alignment Methods

The `align_rows` parameter accepts:
- `'mean'`: Subtract mean value of each row (default)
- `'median'`: Subtract median value of each row
- `'match_height'`: Match row heights at edges
- `None`: Skip row alignment

## Notes

1. All .gwy files are saved in the same directory as the original .sxm files
2. Metadata from the original .sxm files is preserved in the .gwy files
3. The output .gwy files can be opened with Gwyddion or processed further using the gwyfile module
4. Processing is non-destructive - original .sxm files are not modified